In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
from matplotlib.gridspec import GridSpec

In [29]:
# ── Physical parameters (nm) ──────────────────────────────────────────────────
WALL_LO = 0.0
WALL_HI = 225.0
RADIUS  = 50.0
STEP_STD = 1

X_LO, X_HI = 0.0, 300.0       # channel x-extent (nm)

# Wavevectors (units: 1/nm  and  rad/nm respectively)
KX = 2 * np.pi / (405.0/1.33)         # spatial frequency along dummy grid
KY = 2 * np.pi / (405.0/1.33)        # coupling of y_particle → phase shift

# Dummy spatial grid (think of it as a fixed detector array or a k-space axis)
N_GRID = 100
XI     = np.linspace(-1000, 1000, N_GRID)   # nm, arbitrary extent 4 * np.pi / KX

N_FRAMES = 300
INTERVAL = 0.5 # ms

#RNG = np.random.default_rng(2)

# ── Styling ───────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 20,
})

BG    = "#eeeeee"
C_WALL = "#0077ff"
C_PART = "#FC782C"
C_FUNC = "#FC782C"
C_AX   = "#595959"

# ── Initial particle state ────────────────────────────────────────────────────
pos = np.array([X_HI / 2 + 25, WALL_HI / 2])   # (x, y) in nm
#trail_x, trail_y = [], []
#TRAIL_LEN = 100

In [3]:
def iPSF_radial(r, k, zpp, E0, phi0, exp_fac):
    rpp = np.sqrt(r**2 + zpp**2) #from particle to the focal plane
    cos_theta = zpp / rpp #cos of scattering angle
    phi_inc = k*zpp #phase shift due to incedent OPD, (note that influence of zf is lumped into phi0)
    phi_sca = k*rpp #phase shift due to return OPD
    fac = np.sqrt(1+cos_theta**2)*1/(k*rpp) #amplitude factor
    Escat = E0*fac #scattering amplitude

    phi_diff=-(phi0+phi_inc+phi_sca) #phase difference

    iPSF = 2*Escat*np.cos(phi_diff) * (np.exp(-np.abs(r)/exp_fac))

    return iPSF

In [ ]:
%matplotlib qt
plt.close('all')
# ── Figure ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(12, 5))
fig.patch.set_facecolor(BG)

gs = GridSpec(1, 2, figure=fig, wspace=0.12,
              left=0.06, right=0.97, top=0.88, bottom=0.12)

ax_part = fig.add_subplot(gs[0])   # left: particle
ax_func = fig.add_subplot(gs[1])   # right: function

for ax in [ax_part, ax_func]:
    ax.set_facecolor(BG)
    for spine in ax.spines.values():
        spine.set_edgecolor(C_AX)
    ax.tick_params(colors=C_AX, labelsize=14)

# ── Left panel setup ──────────────────────────────────────────────────────────
ax_part.set_xlim(X_LO, X_HI)
ax_part.set_ylim(WALL_LO - 30, WALL_HI + 30)
ax_part.set_aspect("equal")
ax_part.set_xlabel("x  (nm)", color=C_AX, fontsize=15)
ax_part.set_ylabel("y  (nm)", color=C_AX, fontsize=15)
ax_part.set_title("Particle trajectory", color=C_AX, fontsize=20)

# Walls
for y_w in [WALL_LO, WALL_HI]:
    ax_part.axhline(y_w, color=C_WALL, lw=2.2, zorder=3)

ax_part.text(X_HI - 5, WALL_HI + 14, "y = 225 nm",
             color=C_WALL, ha="right", fontsize=15)
ax_part.text(X_HI - 5, WALL_LO - 22, "y = 0",
             color=C_WALL, ha="right", fontsize=15)

circle = plt.Circle(pos, RADIUS, color=C_PART, zorder=5, alpha=0.92)
ax_part.add_patch(circle)

#trail_line, = ax_part.plot([], [], color=C_PART, lw=1.0,
#                           alpha=0.25, zorder=4)

# Dashed horizontal marker showing current y
y_marker, = ax_part.plot([X_LO, X_HI], [pos[1], pos[1]],
                         color=C_PART, lw=0.8, ls="--",
                         alpha=0.5, zorder=4)

y_label = ax_part.text(X_LO + 5, pos[1] + 6,
                       f"y = {pos[1]:.1f} nm",
                       color=C_PART, fontsize=15)

# ── Right panel setup ─────────────────────────────────────────────────────────
ax_func.set_xlim(XI[0], XI[-1])
ax_func.set_ylim(-1.35, 1.35)
ax_func.set_xlabel(r"$\xi$  (nm)", color=C_AX, fontsize=15)
ax_func.set_ylabel(r"$f(\xi;\,t)$", color=C_AX, fontsize=15)
ax_func.set_title(
    r"Radial Scattering Profile",
    color=C_AX, fontsize=20)
ax_func.axhline(0, color=C_AX, lw=0.5, ls=":", alpha=0.4)

# Vertical line showing the phase contribution  ky·y
phase_bar, = ax_func.plot([0, 0], [-1.35, 1.35],
                          color=C_PART, lw=1.2, ls="--", alpha=0.7)

func_line, = ax_func.plot([], [], color=C_FUNC, lw=1.8, zorder=5)

phase_text = ax_func.text(
    0.02, 0.93, "", transform=ax_func.transAxes,
    color=C_PART, fontsize=8, va="top")

# ── Super-title ───────────────────────────────────────────────────────────────
#sup = fig.suptitle("frame 0", color="C_AX", fontsize=20, y=0.97)

# ── Dynamics ──────────────────────────────────────────────────────────────────
def clip(p):
    p[1] = np.clip(p[1], WALL_LO + RADIUS, WALL_HI - RADIUS)
    p[0] = np.clip(p[0], X_LO    + RADIUS, X_HI    - RADIUS)
    return p

def update(frame):
    #global pos
    
    # oscillate down the middle
    #clip(pos + RNG.normal(0, STEP_STD, size=2))
    #pos = pos + 1 *np.cos(2*np.pi *frame/10)
    fixed_position = 225/2 +  50*np.cos(2*np.pi *frame/300)
    # Update trails
    #trail_x.append(X_HI/2 + 25)
    #trail_y.append(fixed_position)
    #trail_x.append(pos[0]);  
    #if len(trail_x) > TRAIL_LEN:
    #    trail_x.pop(0);  trail_y.pop(0)

    # Left panel
    #circle.center = (pos[0], pos[1])
    circle.center = (X_HI/2 + 25, fixed_position)           # (X, Y)
    #trail_line.set_data(trail_x, trail_y)
    y_marker.set_data([X_LO, X_HI], [fixed_position, fixed_position])
    y_label.set_position((X_LO + 5, fixed_position + 6))
    y_label.set_text(f"y = {fixed_position:.1f} nm")

    # Right panel: compute the coupled function
    #phase = KY * pos[1]                       # scalar phase from particle y
    #f_vals = np.cos(KX * XI + phase)

    f_vals = iPSF_radial(XI, KX, fixed_position, 0.6, 0, 1000)

    
    func_line.set_data(XI, f_vals)

    # Marker for the accumulated phase  ky·y  mapped onto the x-axis
    #phase_x = (phase % (2 * np.pi)) / KX      # convert phase → x position
    #phase_bar.set_xdata([phase_x, phase_x])
    #phase_text.set_text(
    #    f"$k_y y_p = {phase:.2f}$ rad\n"
    #    f"$y_p = {pos[1]:.1f}$ nm")

    #sup.set_text(f"Particle-field coupling  |  frame {frame}")

    return (circle, y_marker, y_label,
            func_line, phase_bar, phase_text)

ani = FuncAnimation(fig, update,
                    frames=N_FRAMES, interval=INTERVAL, blit=False, repeat=True)

fig.canvas.draw()

ani.save('RadialProfileVisualization.gif') #savefig_kwargs={'transparent': True})
plt.show()

MovieWriter ffmpeg unavailable; using Pillow instead.
